# Full ResNet-20 inference on PYNQ-Z2
Run in the `pynq/` directory on the board with the current `programmable.bit`, matching `programmable.hwh`, Python helpers, and `full_model.json`. The setup uploads the model once. The next cell runs one image with one START and verifies all ten classifier values.

In [ ]:
import json
from pathlib import Path
from pynq_full_model import FullModel, verify

model = json.loads(Path("full_model.json").read_text())
engine = FullModel("programmable.bit", model)
print(f"One-time model upload: {engine.model_load_ms:.3f} ms")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pynq_full_model import CLASSES

case = model["sample"]
result = engine.infer(case["input"])
exact = result["accum"] == case["accum"] and result["scores"] == case["scores"]
prediction = CLASSES[result["prediction"]]

fig = plt.figure(figsize=(12, 8), constrained_layout=True)
grid = fig.add_gridspec(3, 1, height_ratios=[2.6, 1, 1.7])
scores_ax, precision_ax, time_ax = (fig.add_subplot(grid[i]) for i in range(3))
fig.suptitle(f"ResNet-20 on PYNQ-Z2 | Predicted class: {prediction}", fontsize=15)

positions = range(len(CLASSES))
scores_ax.bar([i - 0.19 for i in positions], case["scores"], 0.38,
              label="CPU reference (saved)", color="#4477AA")
scores_ax.bar([i + 0.19 for i in positions], result["scores"], 0.38,
              label="FPGA (measured)", color="#EE7733")
scores_ax.axhline(0, color="#666666", linewidth=0.7)
scores_ax.set_xticks(list(positions))
scores_ax.set_xticklabels(CLASSES, rotation=20, ha="right")
scores_ax.set_ylabel("Class score (logit)")
matched = sum(a == b and x == y for a, b, x, y in
              zip(case["scores"], result["scores"], case["accum"], result["accum"]))
scores_ax.set_title(f"{'PASS' if exact else 'FAIL'} | {matched}/10 exact classifier outputs", loc="left")
scores_ax.legend(frameon=False, ncol=2)

precision = list(model["precision"].items())
colors = {1: "#AA3377", 4: "#228833", 8: "#4477AA"}
labels = {1: "Binary ±1", 4: "INT4", 8: "INT8"}
for i, (layer, bits) in enumerate(precision):
    precision_ax.barh(0, 0.93, left=i, height=0.62, color=colors[bits])
    if bits != 4:
        precision_ax.text(i + 0.46, 0, labels[bits], color="white",
                          ha="center", va="center", fontsize=7)
groups = [(0, "Conv1"), (1, "Stage 1"), (7, "Stage 2"), (13, "Stage 3"), (19, "Linear")]
precision_ax.set_xticks([(start + end) / 2 for (start, _), end in
                         zip(groups, [1, 7, 13, 19, len(precision)])])
precision_ax.set_xticklabels([label for _, label in groups])
precision_ax.set_xlim(0, len(precision))
precision_ax.set_yticks([])
for spine in precision_ax.spines.values():
    spine.set_visible(False)
precision_ax.set_title("Shared weight precision by layer · INT8 activations", loc="left")
precision_ax.legend(handles=[Patch(color=colors[n], label=labels[n]) for n in (4, 8, 1)],
                    frameon=False, loc="lower right", bbox_to_anchor=(1, 1), ncol=3)

names = ["Image upload", "START to done", "Ten-score readback"]
times = [result["image_load_ms"], result["start_to_done_ms"], result["readback_ms"]]
bars = time_ax.barh(names, times, color=["#4477AA", "#228833", "#EE7733"])
for bar, elapsed in zip(bars, times):
    time_ax.annotate(f"{elapsed:.2f} ms", (elapsed, bar.get_y() + bar.get_height() / 2),
                     xytext=(5, 0), textcoords="offset points", va="center")
time_ax.set_xlim(0, max(times) * 1.24)
time_ax.invert_yaxis()
time_ax.set_xlabel("Board time (ms)")
time_ax.set_title(f"One image · {result['total_ms']:.1f} ms end to end · "
                  f"{result['compute_ms']:.1f} ms FPGA compute", loc="left")
plt.show()

assert exact, "CPU and FPGA classifier outputs differ"


## Optional labeled CIFAR-10 evaluation
Export a test subset with `export_full.py --dataset ... --cases-output ...` and copy `cifar10_cases.json` beside this notebook. The same loaded model runs every image.

In [ ]:
cases_file = Path("cifar10_cases.json")
if cases_file.exists():
    evaluation = json.loads(cases_file.read_text())
    print(evaluation["split"])
    verify(engine, evaluation["cases"])
